# AIC 2026 — Batch Shot Boundary Detection + CAPS v3 (T4 ×2 optimized)

Bản này tối ưu **thời gian chạy toàn dataset** mà vẫn giữ pipeline kiểm tra được:

\[
\text{TransNetV2}
\rightarrow
\text{shared sparse DINO timeline}
\rightarrow
\text{OmniShotCut router}
\rightarrow
\text{final shots}
\rightarrow
\text{CAPS}
\rightarrow
\text{batched quality refinement}
\]

Các thay đổi quan trọng:

- DINOv2 chỉ tạo **một sparse feature store dùng chung** cho mỗi video.
- Long-shot router và CAPS tái sử dụng cùng embedding, không encode lại từng shot.
- Candidate của shot ngắn được bổ sung theo batch, chỉ khi thật sự thiếu.
- OmniShot windows được xếp hạng trước khi chạy; default xử lý các vùng khó nhất.
- FFmpeg dùng **input seeking** và encoder nhanh, thay vì decode từ đầu video cho mỗi window.
- Toàn bộ quality refinement được merge thành các interval rồi decode theo batch.
- Không gọi `torch.cuda.empty_cache()` sau từng DINO batch.
- T4 ×2: GPU 0 chạy TransNetV2 + OmniShotCut, GPU 1 chạy DINOv2.
- Có stage timing cho từng video để biết bottleneck thật.

> Lần đầu nên đặt `MAX_VIDEOS = 2`, `RESUME = False`. Khi output đúng, chuyển `MAX_VIDEOS = None`, `RESUME = True`.


In [ ]:
# ============================================================
# CONFIG
# ============================================================
from pathlib import Path

DATASET_ROOT = Path('/kaggle/input/ai_challange_2026')
AUTO_DISCOVER_DATASET_ROOT = True
VIDEO_GLOB = 'Videos_*/video/*.mp4'
MAP_RELATIVE_GLOB = 'map-keyframes-aic25-b1/map-keyframes/*.csv'

OUTPUT_ROOT = Path('/kaggle/working/batch_keyframe_output_v3')
RESUME = True
CLEAR_OUTPUT_WHEN_NOT_RESUMING = True
MAX_VIDEOS: int | None = None
NUM_SHARDS = 1
SHARD_INDEX = 0
VIDEO_ID_REGEX: str | None = None

# T4 x2
REQUIRE_GPU = True
AUTO_DISTRIBUTE_MODELS_ACROSS_GPUS = True
TRANSNET_GPU_INDEX = 0
OMNISHOT_GPU_INDEX = 0
DINO_GPU_INDEX = 1
ENABLE_AMP = True
AMP_MIN_COMPUTE_CAPABILITY_MAJOR = 7
RANDOM_SEED = 42

# Dependency/model
INSTALL_MISSING_PACKAGES = True
INSTALL_OMNISHOTCUT = True
TRANSNET_WEIGHTS_PATH = Path('/kaggle/working/models/transnetv2-pytorch-weights.pth')
TRANSNET_HF_REPO_ID = 'Sn4kehead/TransNetV2'
TRANSNET_HF_FILENAME = 'transnetv2-pytorch-weights.pth'
DINO_MODEL_NAME = 'facebook/dinov2-small'
DINO_BATCH_SIZE = 96
DINO_PREFETCH_BATCHES = 3
DINO_ANALYSIS_SIZE = 224
OMNISHOTCUT_CHECKPOINT_OR_REPO = 'uva-cv-lab/OmniShotCut'
OMNISHOTCUT_FILENAME = 'OmniShotCut_ckpt.pth'
OMNISHOTCUT_MODE = 'default'
OMNISHOTCUT_OVERLAP = 20
REQUIRE_OMNISHOT = True

# Video decode/logging
SUPPRESS_FFMPEG_DECODER_LOGS = True
FFMPEG_THREADS = 0  # 0 = auto
SEQUENTIAL_DECODE_MAX_GAP_SEC = 3.0
QUALITY_ANALYSIS_MAX_SIDE = 720

# Shared sparse timeline. 2 FPS được dùng lại cho DINO router và CAPS shot dài.
BASE_SAMPLE_FPS = 2.0
SHORT_SHOT_SECONDS = 2.0
SHORT_SHOT_SAMPLE_FPS = 6.0
TRANSITION_EXCLUSION_SEC = 0.15

# Runtime profile:
# - exhaustive: giữ nguyên độ bao phủ của prototype, không giới hạn suspicious windows.
# - balanced: nhanh hơn cho full dataset nhưng có thể bỏ một số OmniShot candidate yếu.
RUNTIME_PROFILE = 'balanced'  # 'exhaustive' hoặc 'balanced'

# Shot boundary detection
TRANSNET_STRONG_THRESHOLD = 0.50
TRANSNET_PEAK_MIN_DISTANCE_SEC = 0.20
LOCALIZATION_TOLERANCE_FRAMES = 3
MIN_SHOT_DURATION_SEC = 0.20
LONG_SHOT_DINO_MAD_LAMBDA = 2.0
LONG_SHOT_DINO_MIN_PROMINENCE_MAD = 0.5
MERGE_WINDOW_GAP_SEC = 0.50

if RUNTIME_PROFILE == 'exhaustive':
    TRANSNET_WEAK_THRESHOLD = 0.30
    TRANSNET_WEAK_MIN_PROMINENCE = 0.0
    MAX_WEAK_CANDIDATES_PER_MINUTE: float | None = None
    SUSPICIOUS_WINDOW_BEFORE_SEC = 2.0
    SUSPICIOUS_WINDOW_AFTER_SEC = 2.0
    LONG_SHOT_MIN_SEC = 8.0
    LONG_SHOT_TOPK_ANOMALIES = 2
    MAX_OMNISHOT_WINDOWS_PER_VIDEO: int | None = None
elif RUNTIME_PROFILE == 'balanced':
    TRANSNET_WEAK_THRESHOLD = 0.35
    TRANSNET_WEAK_MIN_PROMINENCE = 0.03
    MAX_WEAK_CANDIDATES_PER_MINUTE = 6.0
    SUSPICIOUS_WINDOW_BEFORE_SEC = 1.5
    SUSPICIOUS_WINDOW_AFTER_SEC = 1.5
    LONG_SHOT_MIN_SEC = 10.0
    LONG_SHOT_TOPK_ANOMALIES = 1
    MAX_OMNISHOT_WINDOWS_PER_VIDEO = 12
else:
    raise ValueError("RUNTIME_PROFILE phải là 'exhaustive' hoặc 'balanced'.")

KEEP_TEMP_OMNI_WINDOWS = False
USE_NVENC_FOR_OMNI_CLIPS = True
OMNI_CPU_PRESET = 'ultrafast'
OMNI_CPU_CRF = 23

# CAPS temporal change: raw signal, không median filter.
LAMBDA_MAD = 1.5
MIN_PEAK_DISTANCE_SEC = 0.75
MIN_PEAK_PROMINENCE_MAD = 0.50
BUDGET_GAMMA = 0.20
K_MAX = 6
DIVERSITY_ALPHA = 0.70

ENABLE_TEXT_RESCUE = True
TEXTNESS_THRESHOLD = 0.22
TEXT_RESCUE_MIN_BUDGET = 2
ENABLE_RARE_STATE_RESCUE = True
RARE_STATE_MAD_LAMBDA = 2.0

# Quality
DARK_PIXEL_THRESHOLD = 30
OVEREXPOSED_PIXEL_THRESHOLD = 245
W_SHARPNESS = 1.00
W_DARK = 0.45
W_OVEREXPOSED = 0.45
W_TRANSITION = 1.00
REFINEMENT_RADIUS_SEC = 0.25
TEXT_REFINEMENT_BACK_SEC = 0.10
TEXT_REFINEMENT_FORWARD_SEC = 1.00
TEXT_REFINEMENT_WEIGHT = 0.55
QUALITY_TIE_TOLERANCE = 0.03
REFINEMENT_MERGE_GAP_SEC = 0.10
REFINEMENT_FULL_SCAN_COVERAGE = 0.30
REFINEMENT_FULL_SCAN_MIN_INTERVALS = 60

# Output
KEYFRAME_MAX_SIDE: int | None = None
JPEG_QUALITY = 94
SAVE_TRANSNET_PREDICTIONS = True
MAKE_TOP_OMNI_BOUNDARY_PLOT = True
CREATE_METADATA_ZIP = True
CREATE_KEYFRAMES_ZIP = False

# map-keyframes QC: arithmetic check toàn bộ, visual check mặc định tắt để tiết kiệm thời gian.
ENABLE_MAP_ALIGNMENT_QC = True
ENABLE_VISUAL_MAP_QC = False
MAP_VISUAL_SAMPLE_COUNT = 4
MAP_TIME_ERROR_WARN_SEC = 0.050
MAP_FRAME_ERROR_WARN = 1
MAP_VISUAL_SIMILARITY_WARN = 0.90

assert NUM_SHARDS >= 1
assert 0 <= SHARD_INDEX < NUM_SHARDS
assert abs(BASE_SAMPLE_FPS - 2.0) < 1e-6, 'CAPS v3 hiện tối ưu shared timeline cho 2 FPS.'


In [ ]:
# ============================================================
# DEPENDENCIES
# ============================================================
import importlib.util
import shutil
import socket
import subprocess
import sys


def module_exists(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


def internet_available(host: str = 'pypi.org', port: int = 443, timeout: float = 3.0) -> bool:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


def pip_install(packages: list[str], label: str) -> None:
    if not packages:
        return
    print(f'Installing {label}: {" ".join(packages)}')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', *packages],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f'Không cài được {label}.\n{result.stderr[-3000:]}')


HAS_INTERNET = internet_available()
print('Internet:', HAS_INTERNET)
print('ffmpeg:', shutil.which('ffmpeg'))
print('ffprobe:', shutil.which('ffprobe'))

required = {
    'cv2': 'opencv-python-headless',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'scipy': 'scipy',
    'tqdm': 'tqdm',
    'matplotlib': 'matplotlib',
    'PIL': 'Pillow',
    'torch': 'torch',
    'torchvision': 'torchvision',
    'transformers': 'transformers',
    'huggingface_hub': 'huggingface_hub',
    'transnetv2_pytorch': 'transnetv2-pytorch',
}
missing = [pkg for mod, pkg in required.items() if not module_exists(mod)]
if missing:
    if not INSTALL_MISSING_PACKAGES or not HAS_INTERNET:
        raise RuntimeError(f'Thiếu package: {missing}')
    pip_install(missing, 'core packages')

if INSTALL_OMNISHOTCUT and not module_exists('omnishotcut'):
    if not HAS_INTERNET:
        raise RuntimeError('OmniShotCut chưa cài và Internet đang tắt.')
    pip_install(
        ['git+https://github.com/UVA-Computer-Vision-Lab/OmniShotCut.git'],
        'OmniShotCut',
    )


In [ ]:
# ============================================================
# IMPORTS, DEVICE, LOGGING
# ============================================================
from __future__ import annotations

import gc
import inspect
import json
import math
import os
import queue
import random
import re
import shutil
import subprocess
import threading
import time
import traceback
import warnings
from bisect import bisect_left, bisect_right
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Sequence

if SUPPRESS_FFMPEG_DECODER_LOGS:
    os.environ.setdefault('OPENCV_FFMPEG_LOGLEVEL', '-8')
    os.environ.setdefault('OPENCV_LOG_LEVEL', 'SILENT')
    os.environ.setdefault('OPENCV_VIDEOIO_DEBUG', '0')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import FileLink, Image, display
from scipy.signal import find_peaks
from tqdm.auto import tqdm
from transformers import AutoModel
from transformers.utils import logging as hf_logging

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='.*TypedStorage is deprecated.*')
warnings.filterwarnings('ignore', message='.*xFormers.*')
hf_logging.set_verbosity_error()
try:
    cv2.setLogLevel(0 if SUPPRESS_FFMPEG_DECODER_LOGS else 3)
except Exception:
    pass

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError('Không có CUDA. Kaggle Settings → Accelerator → GPU rồi restart session.')

GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0


def safe_gpu_index(requested: int) -> int:
    if GPU_COUNT <= 0:
        return 0
    return int(np.clip(requested, 0, GPU_COUNT - 1))


if GPU_COUNT:
    transnet_idx = safe_gpu_index(TRANSNET_GPU_INDEX)
    omni_idx = safe_gpu_index(OMNISHOT_GPU_INDEX)
    dino_idx = safe_gpu_index(DINO_GPU_INDEX) if AUTO_DISTRIBUTE_MODELS_ACROSS_GPUS and GPU_COUNT >= 2 else transnet_idx
    TRANSNET_DEVICE = torch.device(f'cuda:{transnet_idx}')
    OMNISHOT_DEVICE = torch.device(f'cuda:{omni_idx}')
    DINO_DEVICE = torch.device(f'cuda:{dino_idx}')
else:
    TRANSNET_DEVICE = OMNISHOT_DEVICE = DINO_DEVICE = torch.device('cpu')

if DINO_DEVICE.type == 'cuda':
    DINO_COMPUTE_CAPABILITY = torch.cuda.get_device_capability(DINO_DEVICE)
    DINO_AMP_ENABLED = ENABLE_AMP and DINO_COMPUTE_CAPABILITY[0] >= AMP_MIN_COMPUTE_CAPABILITY_MAJOR
else:
    DINO_COMPUTE_CAPABILITY = (0, 0)
    DINO_AMP_ENABLED = False

print('CUDA GPU count:', GPU_COUNT)
for i in range(GPU_COUNT):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}, CC={torch.cuda.get_device_capability(i)}')
print('TransNetV2:', TRANSNET_DEVICE)
print('OmniShotCut:', OMNISHOT_DEVICE)
print('DINOv2:', DINO_DEVICE, 'AMP=', DINO_AMP_ENABLED)

if not RESUME and CLEAR_OUTPUT_WHEN_NOT_RESUMING and OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PER_VIDEO_ROOT = OUTPUT_ROOT / 'per_video'
KEYFRAME_ROOT = OUTPUT_ROOT / 'keyframes'
TEMP_ROOT = OUTPUT_ROOT / '_tmp'
for p in (PER_VIDEO_ROOT, KEYFRAME_ROOT, TEMP_ROOT):
    p.mkdir(parents=True, exist_ok=True)
ERRORS_PATH = OUTPUT_ROOT / 'errors.csv'


def append_error(video_id: str, stage: str, exc: BaseException) -> None:
    pd.DataFrame([{
        'video_id': video_id,
        'stage': stage,
        'error_type': type(exc).__name__,
        'message': str(exc),
        'traceback': traceback.format_exc()[-8000:],
        'time': pd.Timestamp.utcnow().isoformat(),
    }]).to_csv(ERRORS_PATH, mode='a', index=False, header=not ERRORS_PATH.exists())


def clean_cuda_after_video() -> None:
    gc.collect()
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            with torch.cuda.device(i):
                torch.cuda.empty_cache()


def reset_peak_vram() -> None:
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            torch.cuda.reset_peak_memory_stats(i)


def peak_vram_dict() -> dict[str, float]:
    result = {}
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            result[f'gpu_{i}_peak_vram_gib'] = torch.cuda.max_memory_allocated(i) / 1024**3
    return result


In [ ]:
# ============================================================
# DATASET DISCOVERY
# ============================================================
VIDEO_ID_PATTERN = re.compile(r'^(L\d+_V\d+)$', re.IGNORECASE)

@dataclass(frozen=True)
class VideoInfo:
    video_id: str
    path: Path
    fps: float
    frame_count: int
    width: int
    height: int
    duration_sec: float


def resolve_dataset_root(configured: Path) -> Path:
    if configured.exists():
        return configured
    if not AUTO_DISCOVER_DATASET_ROOT:
        raise FileNotFoundError(f'Không tìm thấy DATASET_ROOT: {configured}')
    for directory in Path('/kaggle/input').iterdir():
        if directory.is_dir() and list(directory.glob(VIDEO_GLOB)) and list(directory.glob(MAP_RELATIVE_GLOB)):
            print('[Auto dataset root]', directory)
            return directory
    raise FileNotFoundError('Không tự tìm được dataset phù hợp dưới /kaggle/input.')


def probe_video(path: Path) -> VideoInfo:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f'Không mở được video: {path}')
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    if not np.isfinite(fps) or fps <= 0 or frame_count <= 0:
        raise RuntimeError(f'Metadata video không hợp lệ: fps={fps}, frames={frame_count}')
    return VideoInfo(path.stem, path, fps, frame_count, width, height, frame_count / fps)


DATASET_ROOT = resolve_dataset_root(DATASET_ROOT)
video_paths = sorted(DATASET_ROOT.glob(VIDEO_GLOB))
map_paths = sorted(DATASET_ROOT.glob(MAP_RELATIVE_GLOB))
map_by_id = {p.stem: p for p in map_paths}
rows = []
for path in video_paths:
    if VIDEO_ID_PATTERN.match(path.stem):
        rows.append({'video_id': path.stem, 'video_path': str(path), 'map_path': str(map_by_id.get(path.stem, '')), 'map_present': path.stem in map_by_id})
inventory = pd.DataFrame(rows).sort_values('video_id').reset_index(drop=True)
if VIDEO_ID_REGEX:
    inventory = inventory[inventory['video_id'].str.match(VIDEO_ID_REGEX, na=False)].reset_index(drop=True)
inventory['global_order'] = np.arange(len(inventory))
inventory = inventory[inventory['global_order'] % NUM_SHARDS == SHARD_INDEX].reset_index(drop=True)
if MAX_VIDEOS is not None:
    inventory = inventory.head(MAX_VIDEOS).copy()
inventory.to_csv(OUTPUT_ROOT / 'video_inventory.csv', index=False)
print('Dataset:', DATASET_ROOT)
print('All videos:', len(video_paths), '| Current run:', len(inventory), '| Maps:', len(map_paths))
display(inventory.head(20))


In [ ]:
# ============================================================
# VIDEO I/O + MAP QC
# ============================================================
def frame_to_sec(frame_idx: int, fps: float) -> float:
    return float(frame_idx) / float(fps)


def sec_to_frame(seconds: float, fps: float) -> int:
    return int(round(float(seconds) * float(fps)))


def clamp_frame(frame_idx: int, frame_count: int) -> int:
    return int(np.clip(frame_idx, 0, max(frame_count - 1, 0)))


def open_capture(path: Path) -> cv2.VideoCapture:
    cap = cv2.VideoCapture(str(path), cv2.CAP_FFMPEG)
    if not cap.isOpened():
        cap = cv2.VideoCapture(str(path))
    return cap


def read_frame_by_index(video_path: Path, frame_idx: int) -> np.ndarray | None:
    cap = open_capture(video_path)
    if not cap.isOpened():
        return None
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame = cap.read()
    cap.release()
    return frame if ok and frame is not None else None


def read_frame_by_time(video_path: Path, time_sec: float) -> tuple[np.ndarray | None, float]:
    cap = open_capture(video_path)
    if not cap.isOpened():
        return None, float('nan')
    cap.set(cv2.CAP_PROP_POS_MSEC, float(time_sec) * 1000.0)
    ok, frame = cap.read()
    actual_sec = float(cap.get(cv2.CAP_PROP_POS_MSEC)) / 1000.0
    cap.release()
    return (frame if ok and frame is not None else None), actual_sec


def split_decode_runs(indices: Sequence[int], max_gap_frames: int) -> list[tuple[int, int, set[int]]]:
    values = sorted(set(int(x) for x in indices))
    if not values:
        return []
    groups = [[values[0]]]
    for idx in values[1:]:
        if idx - groups[-1][-1] <= max_gap_frames:
            groups[-1].append(idx)
        else:
            groups.append([idx])
    return [(g[0], g[-1], set(g)) for g in groups]


def read_frames_by_index(video_path: Path, frame_indices: Sequence[int], fps: float) -> dict[int, np.ndarray]:
    requested = sorted(set(int(x) for x in frame_indices))
    result: dict[int, np.ndarray] = {}
    max_gap = max(1, int(round(SEQUENTIAL_DECODE_MAX_GAP_SEC * fps)))
    for start, end, targets in split_decode_runs(requested, max_gap):
        cap = open_capture(video_path)
        if not cap.isOpened():
            raise RuntimeError(f'Không mở được video: {video_path}')
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
        idx = start
        while idx <= end:
            ok, frame = cap.read()
            if not ok or frame is None:
                idx += 1
                continue
            if idx in targets:
                result[idx] = frame
            idx += 1
        cap.release()
    return result


def resize_max_side(frame: np.ndarray, max_side: int | None) -> np.ndarray:
    if max_side is None:
        return frame
    h, w = frame.shape[:2]
    if max(h, w) <= max_side:
        return frame
    scale = max_side / max(h, w)
    return cv2.resize(frame, (max(1, round(w * scale)), max(1, round(h * scale))), interpolation=cv2.INTER_AREA)


def visual_seek_similarity(a: np.ndarray, b: np.ndarray) -> float:
    a = cv2.resize(cv2.cvtColor(a, cv2.COLOR_BGR2GRAY), (160, 90), interpolation=cv2.INTER_AREA).astype(np.float32)
    b = cv2.resize(cv2.cvtColor(b, cv2.COLOR_BGR2GRAY), (160, 90), interpolation=cv2.INTER_AREA).astype(np.float32)
    return float(np.clip(1.0 - np.mean(np.abs(a - b)) / 255.0, 0.0, 1.0))


def validate_map_alignment(info: VideoInfo, map_path: Path | None, output_dir: Path) -> dict[str, Any]:
    result = {
        'video_id': info.video_id, 'map_present': bool(map_path and map_path.exists()),
        'map_rows': 0, 'video_fps': info.fps, 'map_fps_median': np.nan,
        'fps_abs_error': np.nan, 'max_abs_time_error_sec': np.nan,
        'mean_abs_time_error_sec': np.nan, 'max_abs_frame_error': np.nan,
        'mean_visual_similarity': np.nan, 'min_visual_similarity': np.nan,
        'alignment_status': 'MISSING',
    }
    if not ENABLE_MAP_ALIGNMENT_QC or not map_path or not map_path.exists():
        return result
    df = pd.read_csv(map_path)
    required = {'pts_time', 'fps', 'frame_idx'}
    if not required.issubset(df.columns):
        raise ValueError(f'{map_path.name} thiếu cột: {sorted(required - set(df.columns))}')
    work = df.copy()
    for col in ['pts_time', 'fps', 'frame_idx']:
        work[col] = pd.to_numeric(work[col], errors='coerce')
    work = work.dropna(subset=['pts_time', 'fps', 'frame_idx']).copy()
    work['frame_idx'] = work['frame_idx'].astype(int)
    work['time_from_video_fps'] = work['frame_idx'] / info.fps
    work['abs_time_error_sec'] = (work['pts_time'] - work['time_from_video_fps']).abs()
    work['expected_frame_from_time'] = np.round(work['pts_time'] * info.fps).astype(int)
    work['abs_frame_error'] = (work['frame_idx'] - work['expected_frame_from_time']).abs()

    visual_rows = []
    if ENABLE_VISUAL_MAP_QC and len(work):
        positions = sorted(set(np.linspace(0, len(work) - 1, min(MAP_VISUAL_SAMPLE_COUNT, len(work)), dtype=int).tolist()))
        for pos in positions:
            row = work.iloc[pos]
            by_idx = read_frame_by_index(info.path, clamp_frame(int(row.frame_idx), info.frame_count))
            by_time, actual = read_frame_by_time(info.path, float(row.pts_time))
            sim = visual_seek_similarity(by_idx, by_time) if by_idx is not None and by_time is not None else np.nan
            visual_rows.append({'video_id': info.video_id, 'row_position': pos, 'frame_idx': int(row.frame_idx), 'pts_time': float(row.pts_time), 'actual_seek_sec': actual, 'visual_similarity': sim})
    samples = pd.DataFrame(visual_rows)
    samples.to_csv(output_dir / 'map_alignment_samples.csv', index=False)
    vis = samples['visual_similarity'].dropna().to_numpy() if len(samples) else np.array([])
    result.update({
        'map_rows': int(len(work)),
        'map_fps_median': float(work['fps'].median()) if len(work) else np.nan,
        'fps_abs_error': abs(float(work['fps'].median()) - info.fps) if len(work) else np.nan,
        'max_abs_time_error_sec': float(work['abs_time_error_sec'].max()) if len(work) else np.nan,
        'mean_abs_time_error_sec': float(work['abs_time_error_sec'].mean()) if len(work) else np.nan,
        'max_abs_frame_error': int(work['abs_frame_error'].max()) if len(work) else np.nan,
        'mean_visual_similarity': float(np.mean(vis)) if vis.size else np.nan,
        'min_visual_similarity': float(np.min(vis)) if vis.size else np.nan,
    })
    fail = result['max_abs_frame_error'] > max(3, MAP_FRAME_ERROR_WARN * 3) or result['max_abs_time_error_sec'] > max(0.2, MAP_TIME_ERROR_WARN_SEC * 4)
    warn = result['max_abs_frame_error'] > MAP_FRAME_ERROR_WARN or result['max_abs_time_error_sec'] > MAP_TIME_ERROR_WARN_SEC or (np.isfinite(result['min_visual_similarity']) and result['min_visual_similarity'] < MAP_VISUAL_SIMILARITY_WARN)
    result['alignment_status'] = 'FAIL' if fail else ('WARN' if warn else 'PASS')
    work.to_csv(output_dir / 'map_alignment_full.csv', index=False)
    (output_dir / 'map_alignment_summary.json').write_text(json.dumps(result, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    return result


In [ ]:
# ============================================================
# QUALITY, TEXTNESS, DINO INPUT
# ============================================================
def robust_minmax(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    if values.size == 0:
        return values
    lo, hi = np.percentile(values, [5, 95])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi - lo < 1e-8:
        return np.full_like(values, 0.5, dtype=np.float32)
    return np.clip((values - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)


def timestamp_near_boundary(timestamp_sec: float, boundary_times: Sequence[float], margin_sec: float) -> bool:
    return any(abs(float(timestamp_sec) - float(x)) <= margin_sec for x in boundary_times)


def letterbox_rgb(frame_bgr: np.ndarray, size: int = DINO_ANALYSIS_SIZE) -> np.ndarray:
    h, w = frame_bgr.shape[:2]
    scale = min(size / max(w, 1), size / max(h, 1))
    nw, nh = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
    resized = cv2.resize(frame_bgr, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((size, size, 3), dtype=np.uint8)
    x, y = (size - nw) // 2, (size - nh) // 2
    canvas[y:y+nh, x:x+nw] = resized
    return cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)


def textness_score(frame_bgr: np.ndarray) -> float:
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    if gray.shape[1] > 640:
        scale = 640 / gray.shape[1]
        gray = cv2.resize(gray, (640, max(1, int(round(gray.shape[0] * scale)))), interpolation=cv2.INTER_AREA)
    grad = cv2.convertScaleAbs(cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3))
    _, binary = cv2.threshold(grad, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (17, 3)))
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    image_area = float(gray.shape[0] * gray.shape[1])
    area_sum, count = 0.0, 0
    for contour in contours:
        _, _, bw, bh = cv2.boundingRect(contour)
        ratio = bw * bh / max(image_area, 1.0)
        aspect = bw / max(bh, 1)
        if 2.0 <= aspect <= 40.0 and 0.0004 <= ratio <= 0.15 and bh >= 5:
            area_sum += ratio
            count += 1
    return float(min(1.0, 4.0 * area_sum + 0.02 * min(count, 10)))


def raw_quality_row(frame_bgr: np.ndarray, timestamp: float, boundary_times: Sequence[float]) -> dict[str, float]:
    analysis = resize_max_side(frame_bgr, QUALITY_ANALYSIS_MAX_SIDE)
    gray = cv2.cvtColor(analysis, cv2.COLOR_BGR2GRAY)
    return {
        'sharpness_laplacian_var': float(cv2.Laplacian(gray, cv2.CV_64F).var()),
        'dark_ratio': float(np.mean(gray < DARK_PIXEL_THRESHOLD)),
        'overexposed_ratio': float(np.mean(gray > OVEREXPOSED_PIXEL_THRESHOLD)),
        'transition_risk': 1.0 if timestamp_near_boundary(timestamp, boundary_times, TRANSITION_EXCLUSION_SEC) else 0.0,
        'textness_score': textness_score(analysis) if ENABLE_TEXT_RESCUE else 0.0,
    }


def finalize_quality(raw_details: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    details = raw_details.copy().reset_index(drop=True)
    if details.empty:
        return np.array([], dtype=np.float32), np.array([], dtype=np.float32), details
    sharp = details['sharpness_laplacian_var'].to_numpy(np.float32)
    sharp_norm = robust_minmax(np.log1p(sharp))
    dark = details['dark_ratio'].to_numpy(np.float32)
    over = details['overexposed_ratio'].to_numpy(np.float32)
    transition = details['transition_risk'].to_numpy(np.float32)
    text = details['textness_score'].to_numpy(np.float32)
    raw = W_SHARPNESS * sharp_norm - W_DARK * dark - W_OVEREXPOSED * over - W_TRANSITION * transition
    quality = 0.05 + 0.95 * robust_minmax(raw)
    details['sharpness_norm'] = sharp_norm
    details['quality_raw'] = raw
    details['quality_score'] = quality
    return quality.astype(np.float32), text, details


In [ ]:
# ============================================================
# MODELS + SHARED SPARSE DINO FEATURE STORE
# ============================================================
from huggingface_hub import hf_hub_download
from transnetv2_pytorch import TransNetV2


def locate_transnet_weights() -> Path:
    if TRANSNET_WEIGHTS_PATH.exists():
        return TRANSNET_WEIGHTS_PATH
    TRANSNET_WEIGHTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    try:
        import transnetv2_pytorch
        candidates = sorted(Path(transnetv2_pytorch.__file__).resolve().parent.rglob('*weights*.pth'))
        if candidates:
            shutil.copy2(candidates[0], TRANSNET_WEIGHTS_PATH)
            return TRANSNET_WEIGHTS_PATH
    except Exception:
        pass
    downloaded = Path(hf_hub_download(repo_id=TRANSNET_HF_REPO_ID, filename=TRANSNET_HF_FILENAME, local_dir=str(TRANSNET_WEIGHTS_PATH.parent)))
    if downloaded.resolve() != TRANSNET_WEIGHTS_PATH.resolve():
        shutil.copy2(downloaded, TRANSNET_WEIGHTS_PATH)
    return TRANSNET_WEIGHTS_PATH


def load_transnet() -> TransNetV2:
    if TRANSNET_DEVICE.type == 'cuda':
        torch.cuda.set_device(TRANSNET_DEVICE.index)
    try:
        model = TransNetV2(device=str(TRANSNET_DEVICE))
    except Exception:
        model = TransNetV2(device='auto')
    model.eval()
    weights = locate_transnet_weights()
    map_location = getattr(model, 'device', TRANSNET_DEVICE)
    try:
        state = torch.load(weights, map_location=map_location, weights_only=True)
    except TypeError:
        state = torch.load(weights, map_location=map_location)
    if isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']
    model.load_state_dict(state)
    print('TransNetV2 loaded:', weights)
    return model


class DINOEncoder:
    def __init__(self, model_name: str, device: torch.device):
        self.device = device
        self.amp = DINO_AMP_ENABLED
        self.model = AutoModel.from_pretrained(model_name).eval().to(device)
        self.mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
        print('DINOv2 loaded:', model_name, '| device=', device, '| amp=', self.amp)

    def _encode_once(self, rgb_uint8: np.ndarray) -> np.ndarray:
        x_cpu = torch.from_numpy(np.ascontiguousarray(rgb_uint8)).permute(0, 3, 1, 2).contiguous()
        if torch.cuda.is_available():
            x_cpu = x_cpu.pin_memory()
        x = x_cpu.to(self.device, non_blocking=True).float().div_(255.0)
        x = (x - self.mean) / self.std
        with torch.inference_mode(), torch.autocast(device_type=self.device.type, dtype=torch.float16, enabled=self.amp):
            output = self.model(pixel_values=x).last_hidden_state[:, 0]
        output = F.normalize(output.float(), p=2, dim=-1)
        result = output.cpu().numpy().astype(np.float32)
        del x, output, x_cpu
        return result

    def encode(self, rgb_uint8: np.ndarray) -> np.ndarray:
        if len(rgb_uint8) == 0:
            return np.empty((0, 0), dtype=np.float32)
        try:
            return self._encode_once(rgb_uint8)
        except RuntimeError as exc:
            if 'out of memory' not in str(exc).lower() or len(rgb_uint8) <= 1:
                raise
            if self.device.type == 'cuda':
                with torch.cuda.device(self.device):
                    torch.cuda.empty_cache()
            middle = len(rgb_uint8) // 2
            return np.concatenate([self.encode(rgb_uint8[:middle]), self.encode(rgb_uint8[middle:])], axis=0)


class OmniShotCutAdapter:
    def __init__(self):
        self.model = None
        self.available = False
        self.error = None
        try:
            if OMNISHOT_DEVICE.type == 'cuda':
                torch.cuda.set_device(OMNISHOT_DEVICE.index)
            import omnishotcut
            print('OmniShotCut load:', inspect.signature(omnishotcut.load))
            self.model = omnishotcut.load(OMNISHOTCUT_CHECKPOINT_OR_REPO, filename=OMNISHOTCUT_FILENAME)
            print('OmniShotCut inference:', inspect.signature(self.model.inference))
            self.available = True
        except Exception as exc:
            self.error = f'{type(exc).__name__}: {exc}'
            if REQUIRE_OMNISHOT:
                raise RuntimeError(f'Không load được OmniShotCut: {self.error}') from exc
            print('[WARN]', self.error)

    def infer(self, clip_path: Path) -> dict[str, Any]:
        if not self.available or self.model is None:
            raise RuntimeError(self.error or 'OmniShotCut unavailable')
        if OMNISHOT_DEVICE.type == 'cuda':
            torch.cuda.set_device(OMNISHOT_DEVICE.index)
        with torch.inference_mode():
            result = self.model.inference(str(clip_path), mode=OMNISHOTCUT_MODE, overlap=OMNISHOTCUT_OVERLAP)
        shot_ranges = result[0] if isinstance(result, tuple) else result
        array = np.asarray(shot_ranges)
        ranges = [[int(a), int(b)] for a, b in array.tolist()] if array.size else []
        boundaries = sorted({int(row[0]) for row in ranges[1:] if int(row[0]) > 0})
        return {'shot_ranges_local': ranges, 'boundaries_local': boundaries, 'confidence': None}


TRANSNET_MODEL = load_transnet()
DINO_ENCODER = DINOEncoder(DINO_MODEL_NAME, DINO_DEVICE)
OMNI_ADAPTER = OmniShotCutAdapter()


def ffmpeg_has_encoder(name: str) -> bool:
    result = subprocess.run(['ffmpeg', '-hide_banner', '-encoders'], capture_output=True, text=True)
    return result.returncode == 0 and name in result.stdout

FFMPEG_HAS_NVENC = USE_NVENC_FOR_OMNI_CLIPS and ffmpeg_has_encoder('h264_nvenc')
print('FFmpeg h264_nvenc:', FFMPEG_HAS_NVENC)


def read_exact(stream, byte_count: int) -> bytes:
    parts, remaining = [], byte_count
    while remaining > 0:
        chunk = stream.read(remaining)
        if not chunk:
            break
        parts.append(chunk)
        remaining -= len(chunk)
    return b''.join(parts)


def ffmpeg_sparse_feature_store(info: VideoInfo, sample_fps: float, boundary_times: Sequence[float] = ()) -> dict[int, dict[str, Any]]:
    '''Decode sparse 224x224 frames through FFmpeg and overlap CPU production with DINO GPU inference.'''
    size = DINO_ANALYSIS_SIZE
    vf = f"fps={sample_fps},scale={size}:{size}:force_original_aspect_ratio=decrease,pad={size}:{size}:(ow-iw)/2:(oh-ih)/2"
    command = ['ffmpeg', '-nostdin', '-hide_banner', '-loglevel', 'error']
    if FFMPEG_THREADS > 0:
        command += ['-threads', str(FFMPEG_THREADS)]
    command += ['-i', str(info.path), '-an', '-sn', '-dn', '-vf', vf, '-pix_fmt', 'rgb24', '-f', 'rawvideo', 'pipe:1']

    q: queue.Queue = queue.Queue(maxsize=max(1, DINO_PREFETCH_BATCHES))
    sentinel = object()

    def producer() -> None:
        proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, bufsize=10**8)
        batch_rgb, batch_indices, batch_quality = [], [], []
        sample_no, last_idx = 0, None
        frame_bytes = size * size * 3
        try:
            assert proc.stdout is not None
            while True:
                raw = read_exact(proc.stdout, frame_bytes)
                if len(raw) != frame_bytes:
                    break
                timestamp = sample_no / sample_fps
                frame_idx = clamp_frame(sec_to_frame(timestamp, info.fps), info.frame_count)
                sample_no += 1
                if frame_idx == last_idx:
                    continue
                last_idx = frame_idx
                rgb = np.frombuffer(raw, dtype=np.uint8).reshape(size, size, 3).copy()
                bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
                batch_rgb.append(rgb)
                batch_indices.append(frame_idx)
                batch_quality.append(raw_quality_row(bgr, frame_idx / info.fps, boundary_times))
                if len(batch_rgb) >= DINO_BATCH_SIZE:
                    q.put(('batch', (np.stack(batch_rgb), batch_indices[:], batch_quality[:])))
                    batch_rgb.clear(); batch_indices.clear(); batch_quality.clear()
            if batch_rgb:
                q.put(('batch', (np.stack(batch_rgb), batch_indices[:], batch_quality[:])))
            proc.stdout.close()
            stderr = proc.stderr.read().decode('utf-8', errors='replace') if proc.stderr else ''
            returncode = proc.wait()
            if returncode != 0:
                q.put(('error', RuntimeError(f'FFmpeg sparse decode failed: {stderr[-3000:]}')))
        except BaseException as exc:
            proc.kill()
            q.put(('error', exc))
        finally:
            q.put(('done', sentinel))

    thread = threading.Thread(target=producer, daemon=True)
    thread.start()
    store: dict[int, dict[str, Any]] = {}
    while True:
        kind, payload = q.get()
        if kind == 'done':
            break
        if kind == 'error':
            thread.join(timeout=5)
            raise payload
        rgb, indices, quality_rows = payload
        embeddings = DINO_ENCODER.encode(rgb)
        for pos, idx in enumerate(indices):
            store[int(idx)] = {'embedding': embeddings[pos], 'raw_quality': quality_rows[pos]}
    thread.join()
    return store


def extract_feature_store_for_indices(info: VideoInfo, indices: Sequence[int], boundary_times: Sequence[float]) -> dict[int, dict[str, Any]]:
    frames = read_frames_by_index(info.path, indices, info.fps)
    valid = sorted(frames)
    store: dict[int, dict[str, Any]] = {}
    for start in range(0, len(valid), DINO_BATCH_SIZE):
        batch_indices = valid[start:start + DINO_BATCH_SIZE]
        rgb = np.stack([letterbox_rgb(frames[idx]) for idx in batch_indices])
        embeddings = DINO_ENCODER.encode(rgb)
        for pos, idx in enumerate(batch_indices):
            store[idx] = {
                'embedding': embeddings[pos],
                'raw_quality': raw_quality_row(frames[idx], idx / info.fps, boundary_times),
            }
    return store


In [ ]:
# ============================================================
# SHOT BOUNDARY DETECTION
# ============================================================
def to_probability_1d(value: Any) -> np.ndarray:
    if hasattr(value, 'detach'):
        value = value.detach().cpu().numpy()
    arr = np.squeeze(np.asarray(value)).astype(np.float32)
    return np.clip(arr.reshape(-1), 0.0, 1.0)


def run_transnet(info: VideoInfo) -> tuple[np.ndarray, pd.DataFrame, list[int]]:
    if TRANSNET_DEVICE.type == 'cuda':
        torch.cuda.set_device(TRANSNET_DEVICE.index)
    with torch.inference_mode():
        lowres, single, _ = TRANSNET_MODEL.predict_video(str(info.path))
    probabilities = to_probability_1d(single)
    del lowres, single
    scenes = np.asarray(TRANSNET_MODEL.predictions_to_scenes(probabilities, threshold=TRANSNET_STRONG_THRESHOLD), dtype=np.int64)
    if scenes.size == 0:
        scenes = np.array([[0, len(probabilities) - 1]], dtype=np.int64)
    scenes[:, 0] = np.clip(scenes[:, 0], 0, len(probabilities) - 1)
    scenes[:, 1] = np.clip(scenes[:, 1], 0, len(probabilities) - 1)
    scenes = scenes[scenes[:, 0] <= scenes[:, 1]]
    shots = pd.DataFrame({'shot_id': np.arange(1, len(scenes)+1), 'start_frame': scenes[:,0], 'end_frame': scenes[:,1]})
    shots['start_sec'] = shots.start_frame / info.fps
    shots['end_sec'] = shots.end_frame / info.fps
    shots['duration_sec'] = (shots.end_frame - shots.start_frame + 1) / info.fps
    boundaries = [int(x) for x in shots.start_frame.iloc[1:].tolist()]
    return probabilities, shots, boundaries


def find_weak_transnet_candidates(probabilities: np.ndarray, strong: Sequence[int], info: VideoInfo) -> list[dict[str, Any]]:
    distance = max(1, sec_to_frame(TRANSNET_PEAK_MIN_DISTANCE_SEC, info.fps))
    peaks, props = find_peaks(probabilities, height=TRANSNET_WEAK_THRESHOLD, prominence=TRANSNET_WEAK_MIN_PROMINENCE, distance=distance)
    rows = []
    for peak, height, prominence in zip(peaks, props['peak_heights'], props['prominences']):
        peak = int(peak)
        if height >= TRANSNET_STRONG_THRESHOLD or any(abs(peak - b) <= LOCALIZATION_TOLERANCE_FRAMES for b in strong):
            continue
        rows.append({'center_frame': peak, 'reason': 'weak_transnet', 'transnet_score': float(height), 'dino_score': 0.0, 'candidate_score': float(height + prominence)})
    if MAX_WEAK_CANDIDATES_PER_MINUTE is not None:
        limit = max(1, int(math.ceil(info.duration_sec / 60.0 * MAX_WEAK_CANDIDATES_PER_MINUTE)))
        rows = sorted(rows, key=lambda x: x['candidate_score'], reverse=True)[:limit]
        rows.sort(key=lambda x: x['center_frame'])
    return rows


def store_indices_in_range(store_indices: list[int], start: int, end: int) -> list[int]:
    left, right = bisect_left(store_indices, start), bisect_right(store_indices, end)
    return store_indices[left:right]


def find_long_shot_dino_candidates(transnet_shots: pd.DataFrame, base_store: dict[int, dict[str, Any]]) -> list[dict[str, Any]]:
    rows, indices = [], sorted(base_store)
    for shot in transnet_shots[transnet_shots.duration_sec >= LONG_SHOT_MIN_SEC].itertuples(index=False):
        local = store_indices_in_range(indices, int(shot.start_frame), int(shot.end_frame))
        if len(local) < 3:
            continue
        emb = np.stack([base_store[idx]['embedding'] for idx in local])
        d = np.zeros(len(local), dtype=np.float32)
        d[1:] = np.clip(1.0 - np.sum(emb[1:] * emb[:-1], axis=1), 0.0, 2.0)
        signal = d[1:]
        baseline = float(np.median(signal)); mad = float(np.median(np.abs(signal - baseline)))
        threshold = baseline + LONG_SHOT_DINO_MAD_LAMBDA * mad
        prominence = max(LONG_SHOT_DINO_MIN_PROMINENCE_MAD * mad, 1e-4)
        peaks, props = find_peaks(d, height=threshold + 1e-12, prominence=prominence)
        ranking = sorted([(float(props['prominences'][k]), int(pos)) for k, pos in enumerate(peaks)], reverse=True)[:LONG_SHOT_TOPK_ANOMALIES]
        for prom, pos in ranking:
            center = (local[pos-1] + local[pos]) // 2 if pos > 0 else local[pos]
            rows.append({'center_frame': center, 'reason': 'long_shot_dino_anomaly', 'transnet_score': 0.0, 'dino_score': float(d[pos]), 'candidate_score': prom})
    return rows


def merge_suspicious_windows(candidates: Sequence[dict[str, Any]], info: VideoInfo) -> pd.DataFrame:
    columns = ['window_id','start_frame','end_frame','center_frames','reasons','max_transnet_score','max_dino_score','router_score','selected_for_omnishot']
    if not candidates:
        return pd.DataFrame(columns=columns)
    before = sec_to_frame(SUSPICIOUS_WINDOW_BEFORE_SEC, info.fps)
    after = sec_to_frame(SUSPICIOUS_WINDOW_AFTER_SEC, info.fps)
    gap = sec_to_frame(MERGE_WINDOW_GAP_SEC, info.fps)
    items = [{**x, 'start_frame': clamp_frame(int(x['center_frame']) - before, info.frame_count), 'end_frame': clamp_frame(int(x['center_frame']) + after, info.frame_count)} for x in candidates]
    items.sort(key=lambda x: (x['start_frame'], x['end_frame']))
    groups, current = [], [items[0]]
    for item in items[1:]:
        if item['start_frame'] <= max(x['end_frame'] for x in current) + gap:
            current.append(item)
        else:
            groups.append(current); current = [item]
    groups.append(current)
    rows = []
    for group in groups:
        tn = max(float(x.get('transnet_score', 0.0) or 0.0) for x in group)
        dino = max(float(x.get('dino_score', 0.0) or 0.0) for x in group)
        reasons = sorted(set(x['reason'] for x in group))
        router_score = 0.70 * min(1.0, tn / max(TRANSNET_STRONG_THRESHOLD, 1e-6)) + 0.30 * min(1.0, dino / 0.50) + (0.08 if len(reasons) > 1 else 0.0)
        rows.append({'start_frame': min(x['start_frame'] for x in group), 'end_frame': max(x['end_frame'] for x in group), 'center_frames': ';'.join(str(int(x['center_frame'])) for x in group), 'reasons': ';'.join(reasons), 'max_transnet_score': tn, 'max_dino_score': dino, 'router_score': router_score})
    df = pd.DataFrame(rows).sort_values(['router_score','start_frame'], ascending=[False,True]).reset_index(drop=True)
    df['selected_for_omnishot'] = True
    if MAX_OMNISHOT_WINDOWS_PER_VIDEO is not None and len(df) > MAX_OMNISHOT_WINDOWS_PER_VIDEO:
        df.loc[MAX_OMNISHOT_WINDOWS_PER_VIDEO:, 'selected_for_omnishot'] = False
    df['window_id'] = np.arange(1, len(df)+1)
    return df[columns]


def export_omni_window(info: VideoInfo, start_frame: int, end_frame: int, output_path: Path) -> str:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    start_sec = start_frame / info.fps
    duration_sec = max(1.0 / info.fps, (end_frame - start_frame + 1) / info.fps)
    base = ['ffmpeg','-y','-nostdin','-hide_banner','-loglevel','error','-ss',f'{start_sec:.6f}','-i',str(info.path),'-t',f'{duration_sec:.6f}','-map','0:v:0','-an','-sn','-dn','-vf',f'fps={info.fps:.6f}','-pix_fmt','yuv420p','-avoid_negative_ts','make_zero']
    commands = []
    if FFMPEG_HAS_NVENC:
        commands.append((base + ['-c:v','h264_nvenc','-preset','p4','-cq','23',str(output_path)], 'nvenc'))
    commands.append((base + ['-c:v','libx264','-preset',OMNI_CPU_PRESET,'-crf',str(OMNI_CPU_CRF),'-threads',str(FFMPEG_THREADS),str(output_path)], 'libx264'))
    errors = []
    for command, encoder in commands:
        result = subprocess.run(command, capture_output=True, text=True)
        if result.returncode == 0 and output_path.exists() and output_path.stat().st_size > 0:
            return encoder
        errors.append(f'{encoder}: {result.stderr[-1000:]}')
    raise RuntimeError('Không export được Omni window: ' + ' | '.join(errors))


def run_omnishot_windows(info: VideoInfo, windows: pd.DataFrame, temp_dir: Path) -> pd.DataFrame:
    columns = ['window_id','window_start_frame','window_end_frame','local_frame','global_frame','global_time_sec','omnishot_confidence','reasons','router_score','clip_encoder','shot_ranges_local']
    selected = windows[windows.selected_for_omnishot].copy() if len(windows) else pd.DataFrame()
    rows = []
    for window in selected.itertuples(index=False):
        clip = temp_dir / f'omni_{int(window.window_id):04d}_{int(window.start_frame)}_{int(window.end_frame)}.mp4'
        encoder = export_omni_window(info, int(window.start_frame), int(window.end_frame), clip)
        result = OMNI_ADAPTER.infer(clip)
        for local in result['boundaries_local']:
            global_frame = clamp_frame(int(window.start_frame) + int(local), info.frame_count)
            rows.append({'window_id': int(window.window_id), 'window_start_frame': int(window.start_frame), 'window_end_frame': int(window.end_frame), 'local_frame': int(local), 'global_frame': global_frame, 'global_time_sec': global_frame / info.fps, 'omnishot_confidence': result['confidence'], 'reasons': window.reasons, 'router_score': float(window.router_score), 'clip_encoder': encoder, 'shot_ranges_local': json.dumps(result['shot_ranges_local'])})
        if not KEEP_TEMP_OMNI_WINDOWS:
            clip.unlink(missing_ok=True)
    return pd.DataFrame(rows, columns=columns)


def merge_final_boundaries(info: VideoInfo, probabilities: np.ndarray, strong: Sequence[int], omni_raw: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    items = [{'frame_idx': int(x), 'source': 'transnet', 'transnet_score': float(probabilities[int(x)]), 'omnishot_confidence': np.nan} for x in strong]
    for row in omni_raw.itertuples(index=False):
        frame = int(row.global_frame)
        local_prob = probabilities[max(0, frame-LOCALIZATION_TOLERANCE_FRAMES):min(len(probabilities), frame+LOCALIZATION_TOLERANCE_FRAMES+1)]
        items.append({'frame_idx': frame, 'source': 'omnishot', 'transnet_score': float(local_prob.max()) if len(local_prob) else 0.0, 'omnishot_confidence': row.omnishot_confidence})
    if not items:
        final = pd.DataFrame(columns=['boundary_id','frame_idx','time_sec','source','transnet_score','omnishot_confidence','nested_inside_transnet_shot'])
        shots = pd.DataFrame([{'shot_id':1,'start_frame':0,'end_frame':info.frame_count-1,'start_sec':0.0,'end_sec':(info.frame_count-1)/info.fps,'duration_sec':info.duration_sec,'start_boundary_source':'video_start','end_boundary_source':'video_end'}])
        return final, shots, pd.DataFrame()
    items.sort(key=lambda x: x['frame_idx'])
    groups = []
    for item in items:
        if not groups or item['frame_idx'] - groups[-1][-1]['frame_idx'] > LOCALIZATION_TOLERANCE_FRAMES:
            groups.append([item])
        else:
            groups[-1].append(item)
    merged, log = [], []
    for group in groups:
        sources = {x['source'] for x in group}
        source = 'both' if {'transnet','omnishot'}.issubset(sources) else ('omnishot' if 'omnishot' in sources else 'transnet')
        frames = [x['frame_idx'] for x in group]
        kept = int(round(np.median(frames))) if source == 'both' else int(max(group, key=lambda x: x['transnet_score'])['frame_idx'] if source == 'transnet' else np.median(frames))
        merged.append({'frame_idx': kept, 'time_sec': kept/info.fps, 'source': source, 'transnet_score': max(x['transnet_score'] for x in group), 'omnishot_confidence': np.nan, 'nested_inside_transnet_shot': source in {'omnishot','both'} and all(abs(kept-b)>LOCALIZATION_TOLERANCE_FRAMES for b in strong)})
        for x in group:
            log.append({'event':'boundary_group','input_frame':x['frame_idx'],'input_source':x['source'],'kept_frame':kept,'final_source':source})
    merged = sorted(merged, key=lambda x: x['frame_idx'])
    min_frames = max(1, sec_to_frame(MIN_SHOT_DURATION_SEC, info.fps))
    kept, previous = [], 0
    for row in merged:
        frame = int(row['frame_idx'])
        if frame <= 0 or frame >= info.frame_count or frame - previous < min_frames:
            log.append({'event':'removed_short_or_invalid','input_frame':frame,'input_source':row['source'],'kept_frame':np.nan,'final_source':''})
            continue
        kept.append(row); previous = frame
    if kept and info.frame_count - kept[-1]['frame_idx'] < min_frames:
        dropped = kept.pop(); log.append({'event':'removed_short_last_shot','input_frame':dropped['frame_idx'],'input_source':dropped['source'],'kept_frame':np.nan,'final_source':''})
    final = pd.DataFrame(kept)
    if len(final):
        final.insert(0, 'boundary_id', np.arange(1, len(final)+1))
    shot_rows, start, start_source = [], 0, 'video_start'
    for boundary in final.itertuples(index=False):
        end = int(boundary.frame_idx)-1
        shot_rows.append({'shot_id':len(shot_rows)+1,'start_frame':start,'end_frame':end,'start_sec':start/info.fps,'end_sec':end/info.fps,'duration_sec':(end-start+1)/info.fps,'start_boundary_source':start_source,'end_boundary_source':boundary.source})
        start, start_source = int(boundary.frame_idx), boundary.source
    shot_rows.append({'shot_id':len(shot_rows)+1,'start_frame':start,'end_frame':info.frame_count-1,'start_sec':start/info.fps,'end_sec':(info.frame_count-1)/info.fps,'duration_sec':(info.frame_count-start)/info.fps,'start_boundary_source':start_source,'end_boundary_source':'video_end'})
    return final, pd.DataFrame(shot_rows), pd.DataFrame(log)


def detect_shots_with_shared_store(info: VideoInfo, output_dir: Path, temp_dir: Path, probabilities: np.ndarray, transnet_shots: pd.DataFrame, strong: Sequence[int], base_store: dict[int, dict[str, Any]]) -> dict[str, Any]:
    weak = find_weak_transnet_candidates(probabilities, strong, info)
    dino = find_long_shot_dino_candidates(transnet_shots, base_store)
    windows = merge_suspicious_windows(weak + dino, info)
    omni_raw = run_omnishot_windows(info, windows, temp_dir)
    final_boundaries, final_shots, merge_log = merge_final_boundaries(info, probabilities, strong, omni_raw)
    transnet_shots.to_csv(output_dir / 'transnet_shots.csv', index=False)
    windows.to_csv(output_dir / 'suspicious_windows.csv', index=False)
    omni_raw.to_csv(output_dir / 'omnishot_boundaries_raw.csv', index=False)
    final_boundaries.to_csv(output_dir / 'final_boundaries.csv', index=False)
    final_shots.to_csv(output_dir / 'final_shots.csv', index=False)
    merge_log.to_csv(output_dir / 'boundary_merge_log.csv', index=False)
    if SAVE_TRANSNET_PREDICTIONS:
        pd.DataFrame({'frame_idx':np.arange(len(probabilities)), 'time_sec':np.arange(len(probabilities))/info.fps, 'transnet_probability':probabilities}).to_csv(output_dir / 'transnet_predictions.csv', index=False)
    return {'candidate_windows':windows,'omni_raw':omni_raw,'final_boundaries':final_boundaries,'final_shots':final_shots}


In [ ]:
# ============================================================
# CAPS SELECTION + BATCHED ORIGINAL-FPS REFINEMENT
# ============================================================
def sample_indices_in_shot(start: int, end: int, fps: float, sample_fps: float) -> list[int]:
    start_sec, end_sec = start/fps, (end+1)/fps
    step = 1.0 / max(min(sample_fps, fps), 1e-8)
    times = np.arange(start_sec + 0.5*step, end_sec, step)
    if len(times) == 0:
        times = np.array([(start_sec + end_sec)/2])
    return list(dict.fromkeys(int(np.clip(sec_to_frame(float(t), fps), start, end)) for t in times))


def build_caps_plan(info: VideoInfo, shots: pd.DataFrame, boundaries: pd.DataFrame, base_store: dict[int, dict[str, Any]]) -> dict[int, list[int]]:
    boundary_times = boundaries.time_sec.astype(float).tolist() if len(boundaries) else []
    base_indices = sorted(base_store)
    plan = {}
    for shot in shots.itertuples(index=False):
        start, end, duration = int(shot.start_frame), int(shot.end_frame), float(shot.duration_sec)
        if duration >= SHORT_SHOT_SECONDS:
            candidates = store_indices_in_range(base_indices, start, end)
        else:
            candidates = sample_indices_in_shot(start, end, info.fps, SHORT_SHOT_SAMPLE_FPS)
        candidates = [idx for idx in candidates if not timestamp_near_boundary(idx/info.fps, boundary_times, TRANSITION_EXCLUSION_SEC)]
        if not candidates:
            candidates = [int(round((start+end)/2))]
        plan[int(shot.shot_id)] = list(dict.fromkeys(candidates))
    return plan


def temporal_peaks_raw(embeddings: np.ndarray, timestamps: np.ndarray) -> dict[str, Any]:
    d = np.zeros(len(embeddings), dtype=np.float32)
    if len(embeddings) > 1:
        d[1:] = np.clip(1.0 - np.sum(embeddings[1:] * embeddings[:-1], axis=1), 0.0, 2.0)
    signal = d[1:] if len(d)>1 else d
    baseline = float(np.median(signal)) if len(signal) else 0.0
    mad = float(np.median(np.abs(signal-baseline))) if len(signal) else 0.0
    threshold = baseline + LAMBDA_MAD*mad
    prominence = max(MIN_PEAK_PROMINENCE_MAD*mad, 1e-4)
    if len(d) >= 3:
        deltas = np.diff(timestamps); deltas = deltas[deltas>1e-8]
        median_dt = float(np.median(deltas)) if len(deltas) else MIN_PEAK_DISTANCE_SEC
        distance_samples = max(1, int(math.ceil(MIN_PEAK_DISTANCE_SEC/max(median_dt,1e-8))))
        peaks, props = find_peaks(d, height=threshold+1e-12, prominence=prominence, distance=distance_samples)
    else:
        peaks, props = np.array([],dtype=int), {'prominences':np.array([])}
    return {'distances':d,'baseline':baseline,'mad':mad,'threshold':threshold,'peaks':peaks.astype(int),'prominence_by_index':{int(i):float(p) for i,p in zip(peaks,props.get('prominences',[]))}}


def adaptive_budget(distances: np.ndarray, count: int, max_textness: float) -> tuple[float,int]:
    signal = distances[1:] if len(distances)>1 else distances
    baseline = float(np.median(signal)) if len(signal) else 0.0
    path = float(np.maximum(0.0, signal-baseline).sum())
    budget = 1 + int(math.floor(path/max(BUDGET_GAMMA,1e-8)+0.5))
    if ENABLE_TEXT_RESCUE and max_textness >= TEXTNESS_THRESHOLD:
        budget = max(budget, TEXT_RESCUE_MIN_BUDGET)
    return path, int(np.clip(budget, 1, min(K_MAX, max(count,1))))


def representative_anchor(embeddings: np.ndarray) -> tuple[int,np.ndarray]:
    centroid = embeddings.mean(axis=0); norm = np.linalg.norm(centroid)
    if norm > 1e-12: centroid /= norm
    return int(np.argmax(embeddings @ centroid)), centroid


def select_caps_frames(embeddings: np.ndarray, timestamps: np.ndarray, quality: np.ndarray, textness: np.ndarray, temporal: dict[str,Any], budget: int, duration: float) -> tuple[list[int],dict[int,str],dict[int,float]]:
    anchor, centroid = representative_anchor(embeddings)
    selected, roles, scores = [anchor], {anchor:'anchor'}, {anchor:float(embeddings[anchor]@centroid)}
    if ENABLE_TEXT_RESCUE and len(selected)<budget and float(np.max(textness))>=TEXTNESS_THRESHOLD:
        for idx in np.argsort(-(textness*quality)):
            idx=int(idx)
            if idx not in selected:
                selected.append(idx); roles[idx]='text'; scores[idx]=float(textness[idx]*quality[idx]); break
    ranking = sorted([(float(temporal['prominence_by_index'].get(int(i),0.0)*quality[int(i)]),int(i)) for i in temporal['peaks']], reverse=True)
    for score,idx in ranking:
        if len(selected)>=budget: break
        if idx not in selected:
            selected.append(idx); roles[idx]='peak'; scores[idx]=score
    if ENABLE_RARE_STATE_RESCUE and len(selected)<budget:
        rarity=np.clip(1.0-embeddings@centroid,0.0,2.0)*quality
        med=float(np.median(rarity)); mad=float(np.median(np.abs(rarity-med))); threshold=med+RARE_STATE_MAD_LAMBDA*mad
        for idx in np.argsort(-rarity):
            idx=int(idx)
            if idx not in selected and rarity[idx]>threshold:
                selected.append(idx); roles[idx]='rare_state'; scores[idx]=float(rarity[idx]); break
    while len(selected)<budget:
        remaining=np.array([i for i in range(len(embeddings)) if i not in selected],dtype=int)
        if not len(remaining): break
        s=np.asarray(selected,dtype=int)
        nearest_visual=np.clip(1.0-embeddings[remaining]@embeddings[s].T,0.0,1.0).min(axis=1)
        nearest_time=np.clip(np.abs(timestamps[remaining,None]-timestamps[s][None,:])/max(duration,1e-8),0.0,1.0).min(axis=1)
        combined=quality[remaining]*(DIVERSITY_ALPHA*nearest_visual+(1-DIVERSITY_ALPHA)*nearest_time)
        best=int(np.argmax(combined)); idx=int(remaining[best])
        selected.append(idx); roles[idx]='diversity'; scores[idx]=float(combined[best])
    return selected,roles,scores


def refinement_window(info: VideoInfo, center: float, role: str) -> tuple[int,int]:
    if role=='text':
        start=max(0.0,center-TEXT_REFINEMENT_BACK_SEC); end=min(info.duration_sec,center+TEXT_REFINEMENT_FORWARD_SEC)
    else:
        start=max(0.0,center-REFINEMENT_RADIUS_SEC); end=min(info.duration_sec,center+REFINEMENT_RADIUS_SEC)
    return clamp_frame(math.floor(start*info.fps),info.frame_count), clamp_frame(math.ceil(end*info.fps),info.frame_count)


def merge_refinement_jobs(jobs: list[dict[str,Any]], info: VideoInfo) -> list[dict[str,Any]]:
    gap=max(0,sec_to_frame(REFINEMENT_MERGE_GAP_SEC,info.fps))
    ordered=sorted(jobs,key=lambda x:(x['refine_start'],x['refine_end']))
    groups=[]
    for job in ordered:
        if not groups or job['refine_start']>groups[-1]['end']+gap:
            groups.append({'start':job['refine_start'],'end':job['refine_end'],'jobs':[job]})
        else:
            groups[-1]['end']=max(groups[-1]['end'],job['refine_end']); groups[-1]['jobs'].append(job)
    return groups


def iter_interval_frames(info: VideoInfo, groups: list[dict[str,Any]]):
    covered=sum(g['end']-g['start']+1 for g in groups)
    full_scan = covered/max(info.frame_count,1) >= REFINEMENT_FULL_SCAN_COVERAGE or len(groups)>=REFINEMENT_FULL_SCAN_MIN_INTERVALS
    if full_scan:
        cap=open_capture(info.path)
        if not cap.isOpened(): raise RuntimeError(f'Không mở được {info.path}')
        group_idx=0; current=groups[group_idx] if groups else None; frames={}; idx=0
        while current is not None and idx<=groups[-1]['end']:
            ok,frame=cap.read()
            if not ok or frame is None:
                idx+=1; continue
            if current['start']<=idx<=current['end']: frames[idx]=frame
            if idx>=current['end']:
                yield current,frames
                group_idx+=1; current=groups[group_idx] if group_idx<len(groups) else None; frames={}
            idx+=1
        cap.release()
    else:
        for group in groups:
            cap=open_capture(info.path)
            if not cap.isOpened(): raise RuntimeError(f'Không mở được {info.path}')
            cap.set(cv2.CAP_PROP_POS_FRAMES,int(group['start']))
            frames={}
            for idx in range(int(group['start']),int(group['end'])+1):
                ok,frame=cap.read()
                if ok and frame is not None: frames[idx]=frame
            cap.release()
            yield group,frames


def resize_for_output(frame: np.ndarray) -> np.ndarray:
    return resize_max_side(frame, KEYFRAME_MAX_SIDE)


def batch_refine_save(info: VideoInfo, jobs: list[dict[str,Any]], boundary_times: Sequence[float], keyframe_dir: Path) -> list[dict[str,Any]]:
    for job in jobs:
        job['refine_start'],job['refine_end']=refinement_window(info,job['original_timestamp_sec'],job['selection_role'])
    groups=merge_refinement_jobs(jobs,info)
    metadata=[]; used_by_shot: dict[int,set[int]]={}
    for group,frames in tqdm(iter_interval_frames(info,groups),total=len(groups),desc=f'{info.video_id} refine',leave=False):
        for job in group['jobs']:
            indices=[idx for idx in range(job['refine_start'],job['refine_end']+1) if idx in frames]
            if not indices:
                fallback=read_frame_by_index(info.path,int(job['candidate_frame_idx']))
                if fallback is None: continue
                indices=[int(job['candidate_frame_idx'])]; local_frames=[fallback]
            else:
                local_frames=[frames[idx] for idx in indices]
            rows=[raw_quality_row(frame,idx/info.fps,boundary_times) for frame,idx in zip(local_frames,indices)]
            quality,textness,_=finalize_quality(pd.DataFrame(rows))
            score=quality.copy()
            if job['selection_role']=='text': score=score+TEXT_REFINEMENT_WEIGHT*textness
            best=float(np.max(score)); preferred=np.where(score>=best-QUALITY_TIE_TOLERANCE)[0].tolist()
            preferred.sort(key=lambda p:(abs(indices[p]/info.fps-job['original_timestamp_sec']),-float(score[p])))
            order=preferred+[p for p in np.argsort(-score).tolist() if p not in preferred]
            used=used_by_shot.setdefault(int(job['shot_id']),set())
            chosen=next((p for p in order if indices[p] not in used),order[0]); used.add(indices[chosen])
            refined_idx=indices[chosen]; refined_time=refined_idx/info.fps; frame=resize_for_output(local_frames[chosen])
            filename=f"{info.video_id}_s{int(job['shot_id']):04d}_k{int(job['selection_rank']):02d}_{job['selection_role']}_t{refined_time:010.3f}_f{refined_idx:08d}.jpg"
            output_path=keyframe_dir/filename
            if not cv2.imwrite(str(output_path),frame,[int(cv2.IMWRITE_JPEG_QUALITY),JPEG_QUALITY]):
                raise RuntimeError(f'Không ghi được {output_path}')
            row={k:v for k,v in job.items() if k not in {'embedding','refine_start','refine_end'}}
            row.update({'refined_frame_idx':refined_idx,'refined_timestamp_sec':refined_time,'refined_quality_score':float(quality[chosen]),'refined_textness_score':float(textness[chosen]),'output_path':str(output_path)})
            metadata.append(row)
    return metadata


def extract_caps_keyframes(info: VideoInfo, shots: pd.DataFrame, boundaries: pd.DataFrame, base_store: dict[int,dict[str,Any]], output_dir: Path, keyframe_dir: Path) -> tuple[pd.DataFrame,pd.DataFrame]:
    boundary_times=boundaries.time_sec.astype(float).tolist() if len(boundaries) else []
    keyframe_dir.mkdir(parents=True,exist_ok=True)
    plan=build_caps_plan(info,shots,boundaries,base_store)
    missing=sorted({idx for values in plan.values() for idx in values if idx not in base_store})
    if missing:
        base_store.update(extract_feature_store_for_indices(info,missing,boundary_times))
    jobs=[]; shot_rows=[]
    for shot in tqdm(shots.itertuples(index=False),total=len(shots),desc=f'{info.video_id} CAPS select',leave=False):
        indices=[idx for idx in plan[int(shot.shot_id)] if idx in base_store]
        if not indices: continue
        embeddings=np.stack([base_store[idx]['embedding'] for idx in indices])
        raw=pd.DataFrame([base_store[idx]['raw_quality'] for idx in indices])
        quality,textness,details=finalize_quality(raw)
        timestamps=np.asarray(indices,dtype=np.float32)/info.fps
        temporal=temporal_peaks_raw(embeddings,timestamps)
        path,budget=adaptive_budget(temporal['distances'],len(indices),float(np.max(textness)))
        selected,roles,scores=select_caps_frames(embeddings,timestamps,quality,textness,temporal,budget,float(shot.duration_sec))
        selected=sorted(selected,key=lambda p:timestamps[p])
        for rank,pos in enumerate(selected,1):
            detail=details.iloc[pos]
            jobs.append({'video_id':info.video_id,'video_path':str(info.path),'shot_id':int(shot.shot_id),'shot_start_frame':int(shot.start_frame),'shot_end_frame':int(shot.end_frame),'shot_start_sec':float(shot.start_sec),'shot_end_sec':float(shot.end_sec),'shot_duration_sec':float(shot.duration_sec),'candidate_count':len(indices),'path_length':path,'budget_k':budget,'detected_peak_count':len(temporal['peaks']),'peak_threshold':temporal['threshold'],'peak_mad':temporal['mad'],'selection_rank':rank,'selection_role':roles[pos],'selection_score':scores[pos],'candidate_position':pos,'candidate_frame_idx':int(indices[pos]),'original_timestamp_sec':float(timestamps[pos]),'candidate_quality_score':float(quality[pos]),'candidate_textness_score':float(textness[pos]),'sharpness_laplacian_var':float(detail.sharpness_laplacian_var),'dark_ratio':float(detail.dark_ratio),'overexposed_ratio':float(detail.overexposed_ratio)})
        shot_rows.append({'video_id':info.video_id,'shot_id':int(shot.shot_id),'start_frame':int(shot.start_frame),'end_frame':int(shot.end_frame),'duration_sec':float(shot.duration_sec),'candidate_count':len(indices),'peak_count':len(temporal['peaks']),'path_length':path,'budget_k':budget,'selected_count':len(selected),'max_textness':float(np.max(textness))})
    metadata=pd.DataFrame(batch_refine_save(info,jobs,boundary_times,keyframe_dir))
    shot_summary=pd.DataFrame(shot_rows)
    metadata.to_csv(output_dir/'keyframes_metadata.csv',index=False)
    shot_summary.to_csv(output_dir/'caps_shot_summary.csv',index=False)
    return metadata,shot_summary


In [ ]:
# ============================================================
# PROCESS ONE VIDEO + BATCH LOOP
# ============================================================
def success_path(video_id: str) -> Path:
    return PER_VIDEO_ROOT/video_id/'_SUCCESS.json'


def process_one_video(video_path: Path, map_path: Path | None) -> dict[str,Any]:
    reset_peak_vram()
    info=probe_video(video_path)
    video_dir=PER_VIDEO_ROOT/info.video_id; video_dir.mkdir(parents=True,exist_ok=True)
    keyframe_dir=KEYFRAME_ROOT/info.video_id; keyframe_dir.mkdir(parents=True,exist_ok=True)
    temp_dir=TEMP_ROOT/info.video_id
    if temp_dir.exists(): shutil.rmtree(temp_dir)
    temp_dir.mkdir(parents=True,exist_ok=True)
    stage={}; total=time.perf_counter()

    t=time.perf_counter(); map_summary=validate_map_alignment(info,map_path,video_dir); stage['map_qc_sec']=time.perf_counter()-t
    t=time.perf_counter(); probabilities,transnet_shots,strong=run_transnet(info); stage['transnet_sec']=time.perf_counter()-t
    t=time.perf_counter(); base_store=ffmpeg_sparse_feature_store(info,BASE_SAMPLE_FPS); stage['shared_dino_store_sec']=time.perf_counter()-t
    t=time.perf_counter(); shot_result=detect_shots_with_shared_store(info,video_dir,temp_dir,probabilities,transnet_shots,strong,base_store); stage['omni_router_and_merge_sec']=time.perf_counter()-t
    t=time.perf_counter(); keyframes,caps_shots=extract_caps_keyframes(info,shot_result['final_shots'],shot_result['final_boundaries'],base_store,video_dir,keyframe_dir); stage['caps_and_refinement_sec']=time.perf_counter()-t
    stage['total_sec']=time.perf_counter()-total

    final=shot_result['final_boundaries']; omni_raw=shot_result['omni_raw']; windows=shot_result['candidate_windows']
    summary={
        'video_id':info.video_id,'video_path':str(info.path),'map_path':str(map_path) if map_path else '',
        'fps':info.fps,'frame_count':info.frame_count,'duration_sec':info.duration_sec,'width':info.width,'height':info.height,
        'transnet_boundary_count':len(strong),'suspicious_window_count':len(windows),'omnishot_selected_window_count':int(windows.selected_for_omnishot.sum()) if len(windows) else 0,
        'omnishot_raw_count':len(omni_raw),'omnishot_unique_final_count':int(final.source.isin(['omnishot','both']).sum()) if len(final) else 0,
        'omnishot_only_final_count':int((final.source=='omnishot').sum()) if len(final) else 0,'both_final_count':int((final.source=='both').sum()) if len(final) else 0,
        'nested_omnishot_count':int(final.nested_inside_transnet_shot.sum()) if len(final) else 0,'final_boundary_count':len(final),'final_shot_count':len(shot_result['final_shots']),
        'keyframe_count':len(keyframes),'keyframes_per_minute':len(keyframes)/max(info.duration_sec/60,1e-8),
        'map_alignment_status':map_summary['alignment_status'],'map_max_time_error_sec':map_summary['max_abs_time_error_sec'],'map_max_frame_error':map_summary['max_abs_frame_error'],'map_min_visual_similarity':map_summary['min_visual_similarity'],
        **stage,**peak_vram_dict(),'completed_at':pd.Timestamp.utcnow().isoformat(),
    }
    (video_dir/'stage_times.json').write_text(json.dumps(stage,indent=2),encoding='utf-8')
    (video_dir/'summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
    success_path(info.video_id).write_text(json.dumps(summary,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
    shutil.rmtree(temp_dir,ignore_errors=True)
    print(f"[{info.video_id}] total={stage['total_sec']/60:.1f}m | transnet={stage['transnet_sec']/60:.1f}m | shared-DINO={stage['shared_dino_store_sec']/60:.1f}m | Omni={stage['omni_router_and_merge_sec']/60:.1f}m | CAPS={stage['caps_and_refinement_sec']/60:.1f}m")
    return summary


summary_rows=[]
for row in tqdm(inventory.itertuples(index=False),total=len(inventory),desc='All videos'):
    video_id=row.video_id; video_path=Path(row.video_path); map_path=Path(row.map_path) if row.map_path else None
    if RESUME and success_path(video_id).exists():
        try:
            summary_rows.append(json.loads(success_path(video_id).read_text(encoding='utf-8'))); tqdm.write(f'[resume] {video_id}'); continue
        except Exception: pass
    try:
        summary=process_one_video(video_path,map_path); summary_rows.append(summary)
        tqdm.write(f"[done] {video_id}: {summary['total_sec']/60:.1f}m, shots={summary['final_shot_count']}, keyframes={summary['keyframe_count']}, omni={summary['omnishot_unique_final_count']}")
    except Exception as exc:
        append_error(video_id,'process_one_video',exc); tqdm.write(f'[error] {video_id}: {type(exc).__name__}: {exc}')
    finally:
        clean_cuda_after_video()

dataset_summary=pd.DataFrame(summary_rows)
dataset_summary.to_csv(OUTPUT_ROOT/'dataset_summary.csv',index=False)
print('Completed/resumed:',len(dataset_summary))
if len(dataset_summary):
    timing_cols=['video_id','duration_sec','total_sec','transnet_sec','shared_dino_store_sec','omni_router_and_merge_sec','caps_and_refinement_sec']
    display(dataset_summary[[c for c in timing_cols if c in dataset_summary.columns]].head(20))


In [ ]:
# ============================================================
# AGGREGATE + TOP OMNISHOT BOUNDARY PLOT
# ============================================================
def concatenate_csv(pattern: str, output: Path) -> pd.DataFrame:
    frames=[]
    for path in sorted(PER_VIDEO_ROOT.glob(pattern)):
        try:
            df=pd.read_csv(path)
            if len(df):
                if 'video_id' not in df.columns: df.insert(0,'video_id',path.parent.name)
                frames.append(df)
        except Exception: pass
    result=pd.concat(frames,ignore_index=True) if frames else pd.DataFrame(); result.to_csv(output,index=False); return result

all_keyframes=concatenate_csv('*/keyframes_metadata.csv',OUTPUT_ROOT/'all_keyframes_metadata.csv')
all_boundaries=concatenate_csv('*/final_boundaries.csv',OUTPUT_ROOT/'all_boundaries.csv')
reports=[]
for path in sorted(PER_VIDEO_ROOT.glob('*/map_alignment_summary.json')):
    try: reports.append(json.loads(path.read_text(encoding='utf-8')))
    except Exception: pass
pd.DataFrame(reports).to_csv(OUTPUT_ROOT/'map_alignment_report.csv',index=False)

if len(dataset_summary):
    ranking=dataset_summary.sort_values(['omnishot_unique_final_count','omnishot_raw_count','nested_omnishot_count'],ascending=False).reset_index(drop=True)
    ranking.insert(0,'rank',np.arange(1,len(ranking)+1))
else: ranking=pd.DataFrame()
ranking.to_csv(OUTPUT_ROOT/'omnishot_ranking.csv',index=False)

plot_path=OUTPUT_ROOT/'top_omnishot_boundary_plot.png'
if MAKE_TOP_OMNI_BOUNDARY_PLOT and len(ranking):
    top=ranking.iloc[0]; video_id=str(top.video_id); directory=PER_VIDEO_ROOT/video_id
    predictions=pd.read_csv(directory/'transnet_predictions.csv')
    boundaries=pd.read_csv(directory/'final_boundaries.csv') if (directory/'final_boundaries.csv').exists() else pd.DataFrame()
    tn=boundaries[boundaries.source.isin(['transnet','both'])] if len(boundaries) else pd.DataFrame()
    omni=boundaries[boundaries.source.isin(['omnishot','both'])] if len(boundaries) else pd.DataFrame()
    step=max(1,int(math.ceil(len(predictions)/50000)))
    fig,ax=plt.subplots(figsize=(18,6)); ax.plot(predictions.time_sec.iloc[::step],predictions.transnet_probability.iloc[::step],linewidth=.8,label='TransNetV2 probability')
    ax.axhline(TRANSNET_STRONG_THRESHOLD,linestyle='--',linewidth=1,label=f'Threshold={TRANSNET_STRONG_THRESHOLD}')
    if len(tn): ax.scatter(tn.time_sec,np.full(len(tn),1.02),marker='v',s=30,label='TransNetV2 final boundary',clip_on=False)
    if len(omni): ax.scatter(omni.time_sec,np.full(len(omni),1.08),marker='x',s=38,label='OmniShotCut final boundary',clip_on=False)
    ax.set_ylim(-.02,1.13); ax.set_xlabel('Timestamp (s)'); ax.set_ylabel('TransNetV2 probability')
    ax.set_title(f"{video_id} | Omni unique={int(top.omnishot_unique_final_count)} | TransNet={int(top.transnet_boundary_count)} | final shots={int(top.final_shot_count)}")
    ax.grid(alpha=.2); ax.legend(loc='upper right'); fig.tight_layout(); fig.savefig(plot_path,dpi=170,bbox_inches='tight'); plt.close(fig)
    display(Image(filename=str(plot_path)))
display(ranking.head(20))


In [ ]:
# ============================================================
# CONFIG + ARCHIVE
# ============================================================
CONFIG={k:v for k,v in globals().copy().items() if k.isupper() and k not in {'TRANSNET_MODEL','DINO_ENCODER','OMNI_ADAPTER','TRANSNET_DEVICE','OMNISHOT_DEVICE','DINO_DEVICE'}}
(OUTPUT_ROOT/'run_config.json').write_text(json.dumps(CONFIG,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
if CREATE_METADATA_ZIP:
    staging=OUTPUT_ROOT/'_metadata_archive'; shutil.rmtree(staging,ignore_errors=True); staging.mkdir(parents=True)
    for name in ['run_config.json','video_inventory.csv','dataset_summary.csv','all_keyframes_metadata.csv','all_boundaries.csv','map_alignment_report.csv','errors.csv','omnishot_ranking.csv','top_omnishot_boundary_plot.png']:
        source=OUTPUT_ROOT/name
        if source.exists(): shutil.copy2(source,staging/name)
    archive=Path(shutil.make_archive('/kaggle/working/batch_keyframe_metadata_v3','zip',root_dir=staging)); shutil.rmtree(staging)
    print('Metadata ZIP:',archive); display(FileLink(str(archive)))
if CREATE_KEYFRAMES_ZIP:
    archive=Path(shutil.make_archive('/kaggle/working/all_keyframes_v3','zip',root_dir=KEYFRAME_ROOT)); display(FileLink(str(archive)))
print('Output:',OUTPUT_ROOT); print('Keyframes:',KEYFRAME_ROOT)
display(FileLink(str(OUTPUT_ROOT/'dataset_summary.csv')))


## Cấu hình tốc độ và độ bao phủ

Notebook mặc định dùng `RUNTIME_PROFILE='exhaustive'` để không giới hạn OmniShot candidates. Khi đã kiểm tra output và cần tăng tốc, đổi sang `balanced`:

```python
RUNTIME_PROFILE = 'balanced'
```

Hai profile không thay đổi batching/shared feature store; chúng chỉ thay đổi số vùng gọi OmniShotCut.

Muốn chạy nhiều Kaggle session, chia shard:

```python
NUM_SHARDS = 4
SHARD_INDEX = 0  # lần lượt 0, 1, 2, 3
```

`dataset_summary.csv` có thời gian từng stage. Hãy xem cột nào lớn nhất trước khi tune tiếp.
